# 1. Imports
This section contains all the needed packages for the notebook format.

In [1]:
# =====================================================================
# 1. Imports
# =====================================================================
# Determine the execution environment (Local vs Google Colab) and configure path structures
import sys
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
IN_COLAB = False
print(f"[INFO] Environment: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/CV-project-25-26'
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
    print(f"[INFO] CWD: {os.getcwd()}")

# Verify GPU availability and CUDA setup
import torch
print(f"[INFO] PyTorch  : {torch.__version__}")
print(f"[INFO] CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[INFO] GPU      : {torch.cuda.get_device_name(0)}")
    print(f"[INFO] VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set float32 matrix multiplication precision to high for training acceleration
torch.set_float32_matmul_precision('high')
print("[INFO] float32 matmul precision -> high")


[INFO] Environment: Local
[INFO] PyTorch  : 2.12.0.dev20260408+cu128
[INFO] CUDA     : True
[INFO] GPU      : NVIDIA GeForce RTX 5070 Laptop GPU
[INFO] VRAM     : 8.5 GB
[INFO] float32 matmul precision -> high


In [ ]:
import gc
import time
from pathlib import Path

import lightning as L
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor

# Project modules 
import src.config as cfg
from src.config import (
    DEVICE, ARCH_TYPE,
    SSL_BATCH_SIZE, FT_BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY,
    EPOCHS_PRETRAIN, EPOCHS_FINETUNE,
    SSL_LR, SSL_WEIGHT_DECAY, SSL_PROJ_DIM, SSL_NUM_SLICES, SSL_LAMB,
    LAMBDA_1, LAMBDA_2, LAMBDA_3,
    DATASET_NYU_PATH, DATASET_KITTI_PATH, DATASET_KITTI_PATH_GT,
    MODEL_SAVE_PATH, SSL_CKPT_PATH, FINETUNE_CKPT_PATH,
    NYU_SIZE, KITTI_SIZE,
    OUTPUT_DIR, IMAGES_DIR, JSON_DIR,
)
from src.network    import METERModel
from src.dataset    import NYUDataset, KITTIDataset, build_dataloader
from src.evaluation import (
    evaluate_depth_model,
    geometric_probing_pca,
    geometric_probing_test_set,
    evaluate_pixel_weighted,
)
from src.utils import Timer, save_checkpoint, load_checkpoint, log_metrics_to_csv
from src.train  import LeJEPAMeter, METERFinetune, get_callbacks


# 2. Globals
This section contains useful variables, seeds, and configurations across the whole code.

In [ ]:
# =====================================================================
# 2. Globals
# =====================================================================
# Seed setup to ensure reproducible training and evaluation runs
# Configuration summary 
print("=" * 55)
print("  METER + LeJEPA — Project 7 Configuration")
print("=" * 55)
print(f"  Device          : {DEVICE}")
print(f"  Architecture    : METER-{ARCH_TYPE.upper()}")
print()
print("  [Pre-training — LeJEPA SSL]")
print(f"    Epochs        : {EPOCHS_PRETRAIN}")
print(f"    LR            : {SSL_LR}")
print(f"    Weight decay  : {SSL_WEIGHT_DECAY}")
print(f"    Proj dim      : {SSL_PROJ_DIM}")
print(f"    SIGReg slices : {SSL_NUM_SLICES}")
print(f"    Lambda (SIG)  : {SSL_LAMB}")
print()
print("  [Fine-tuning — Supervised MDE  (paper Sec. IV-A)]")
print(f"    Epochs        : {EPOCHS_FINETUNE}")
print(f"    Batch size SSL: {SSL_BATCH_SIZE}")
print(f"    Batch size FT : {FT_BATCH_SIZE}")
print(f"    LR            : {LEARNING_RATE}   (AdamW, StepLR /10 every 20 ep.)")
print(f"    Weight decay  : {WEIGHT_DECAY}")
print(f"    λ1 (grad)     : {LAMBDA_1}")
print(f"    λ2 (normal)   : {LAMBDA_2}")
print(f"    λ3 (ssim)     : {LAMBDA_3}")
print()
print("  [Paths]")
print(f"    NYU           : {DATASET_NYU_PATH}")
print(f"    KITTI RGB     : {DATASET_KITTI_PATH}")
print(f"    KITTI GT      : {DATASET_KITTI_PATH_GT}")
print(f"    Checkpoints   : {MODEL_SAVE_PATH}")
print("=" * 55)

# Reproducibility seed 
L.seed_everything(42, workers=True)
print("[INFO] Global seed → 42")

# 3. Utils
This section references code support functions. Utility helper functions such as `Timer`, `save_checkpoint`, `load_checkpoint`, `log_metrics_to_csv` are imported from `src.utils` to keep the notebook clean and modular. The helper below selects the best fine-tuning checkpoint by validation RMSE (parsed from the filename), falling back to `last.ckpt`.

In [ ]:
# =====================================================================
# 3. Utils
# =====================================================================
import re

def find_best_ckpt(ckpt_dir: Path) -> Path | None:
    """
    Return the checkpoint with the lowest validation RMSE in `ckpt_dir`.
    Filenames follow get_callbacks(): "best-{epoch}-{val_rmse:.4f}-{dataset}-{arch}.ckpt".
    Falls back to `last.ckpt` when no 'best-*' file is found.
    """
    ckpt_dir = Path(ckpt_dir)
    if not ckpt_dir.exists():
        return None
    best, best_rmse = None, float("inf")
    for p in ckpt_dir.glob("*.ckpt"):
        m = re.search(r"val[/_]rmse=([0-9]+\.[0-9]+)", p.name)
        if m and float(m.group(1)) < best_rmse:
            best, best_rmse = p, float(m.group(1))
    if best is not None:
        return best
    last = ckpt_dir / "last.ckpt"
    return last if last.exists() else None

print("[OK] Utils ready (find_best_ckpt)")


# 4. Data
This section manages dataset loading, preprocessing splits, and DataLoaders building.

In [ ]:
# =====================================================================
# 4. Data
# =====================================================================
# Downstream transfer learning dataset selection: "nyu" | "kitti"
# Select fine-tuning dataset 
# Options: "nyu" | "kitti"
DATASET_NAME = "nyu" 
FT_BATCH = 16 if DATASET_NAME == "kitti" else FT_BATCH_SIZE


print(f"\n[INFO] Selected dataset: {DATASET_NAME.upper()}")

# ------------------------------------------------------------
# 4A — SSL Pre-training DataLoader  (in-domain per dataset)
# ------------------------------------------------------------
if DATASET_NAME == "nyu":
    print("\n[STEP 4A] SSL DataLoader (NYU raw — pretrain mode)")
    _full_ssl = NYUDataset(
        path   = DATASET_NYU_PATH,
        mode   = "pretrain",
        split  = "train",
        kaggle = False,
    )
    from torch.utils.data import random_split
    _gen = torch.Generator().manual_seed(42)
    _n_total = len(_full_ssl)
    _n_val   = max(1, int(0.1 * _n_total))
    _n_train = _n_total - _n_val
    ssl_dataset, ssl_val_dataset = random_split(
        _full_ssl, [_n_train, _n_val], generator=_gen
    )
    print(f"[INFO] SSL split — train: {_n_train:,}  val: {_n_val:,}")
elif DATASET_NAME == "kitti":
    print("\n[STEP 4A] SSL DataLoader (KITTI raw — pretrain mode)")
    kitti_rgb_root = DATASET_KITTI_PATH / "rgb"
    _full_kitti_ssl = KITTIDataset(
        path  = kitti_rgb_root,
        mode  = "pretrain",
        split = "all",
    )
    # Subset random del train: SSL non ha bisogno di tutti i 163k sample
    from torch.utils.data import Subset
    import random as _r
    _r.seed(42)
    _train_ids = _r.sample(range(len(_full_kitti_ssl)), k=10000)
    ssl_dataset = Subset(_full_kitti_ssl, _train_ids)
    print(f"[INFO] SSL train subset: {len(ssl_dataset):,} samples (from {len(_full_kitti_ssl):,} total)")
    # Use the same KITTI raw as val (SSL is self-supervised, no labels needed)
    ssl_val_dataset = KITTIDataset(
        path  = kitti_rgb_root,
        mode  = "pretrain",
        split = "all",
    )
    from torch.utils.data import Subset
    import random as _r
    _r.seed(42)
    _val_ids = _r.sample(range(len(_full_kitti_ssl)), k=2000) 
    ssl_val_dataset = Subset(_full_kitti_ssl, _val_ids)
else:
    raise ValueError(f"DATASET_NAME must be 'nyu' or 'kitti' for SSL, got '{DATASET_NAME}'")

ssl_loader     = build_dataloader(ssl_dataset,     batch_size=SSL_BATCH_SIZE, shuffle=True,  drop_last=True)
ssl_val_loader = build_dataloader(ssl_val_dataset, batch_size=SSL_BATCH_SIZE, shuffle=False, drop_last=True) 
print(f"[INFO] SSL dataset  : {len(ssl_dataset):,} samples")
print(f"[INFO] SSL batches  : {len(ssl_loader):,}")
print(f"[INFO] SSL val      : {len(ssl_val_dataset):,} samples")

# ------------------------------------------------------------
# 4B — Fine-tuning DataLoaders
# ------------------------------------------------------------

if DATASET_NAME in ("nyu", "both"):
    print("\n[STEP 4B] NYU Fine-tuning DataLoaders")

    # Train: 50K subset — paper Section IV-B
    nyu_train = NYUDataset(
        path   = DATASET_NYU_PATH,
        mode   = "finetune",
        split  = "train",
        kaggle = False,
    )
    # Test: 654 pairs from the official .mat — paper Section IV-B
    nyu_test = NYUDataset(
        path   = DATASET_NYU_PATH,
        mode   = "finetune",
        split  = "test",
        kaggle = False,
    )
    # Sanity check: paper uses ~50K train samples (Section IV-B).
    # If you are below ~40K you are likely on the .mat (795 samples) — fix the dataset path.
    n_train = len(nyu_train)
    print(f"[CHECK] NYU train samples: {n_train:,}")
    if n_train < 40000:
        print(f"[WARN] NYU train has only {n_train} samples — the paper uses ~50K.")
        print("[WARN] Expected if you are on the .mat (1449 total). Consider switching to "
              "Kaggle H5 (kaggle=True) or the 50K jpg/png pairs format to match the paper.")
    from src.utils import random_subset
    nyu_train_subset = random_subset(nyu_train, subset_size=10000, seed=42)
    nyu_train_loader = build_dataloader(nyu_train_subset, batch_size=FT_BATCH_SIZE, shuffle=True,  drop_last=True)
    nyu_test_loader  = build_dataloader(nyu_test,  batch_size=FT_BATCH_SIZE, shuffle=False)

    print(f"[INFO] NYU train    : {len(nyu_train_subset):,} samples  ({len(nyu_train_loader):,} batches)")
    print(f"[INFO] NYU test     : {len(nyu_test):,}  samples  ({len(nyu_test_loader):,}  batches)")

    # Sanity check: verify batch shape
    _rgb, _depth = next(iter(nyu_train_loader))
    print(f"[INFO] NYU batch    : RGB {tuple(_rgb.shape)}  Depth {tuple(_depth.shape)}")
    print(f"[INFO] Depth range  : [{_depth.min():.1f}, {_depth.max():.1f}] cm")
    assert _rgb.shape[1:] == (3, 192, 256),   f"Expected (3,192,256) — got {_rgb.shape[1:]}"
    assert _depth.shape[1:] == (1, 192, 256), f"Expected (1,192,256) — got {_depth.shape[1:]}"
    print("[OK] NYU shape verified")

if DATASET_NAME in ("kitti", "both"):
    print("\n[STEP 4B] KITTI Fine-tuning DataLoaders")

    # Expected structure by KITTIDataset (see dataset.py):
    #   path    = <kitti_root>/rgb/
    #   gt_root = <kitti_root>/depth/data_depth_annotated/
    kitti_rgb_root = DATASET_KITTI_PATH / "rgb"
    kitti_gt_root  = DATASET_KITTI_PATH_GT / "depth" 

    kitti_train = KITTIDataset(
        path     = kitti_rgb_root,
        gt_root  = kitti_gt_root,
        mode     = "finetune",
        split    = "train",
    )
    kitti_val = KITTIDataset(
        path     = kitti_rgb_root,
        gt_root  = kitti_gt_root,
        mode     = "finetune",
        split    = "val",
    )

    from src.utils import random_subset
    kitti_train_subset = random_subset(kitti_train, subset_size=10000, seed=42)
    kitti_train_loader = build_dataloader(kitti_train_subset, batch_size=FT_BATCH, shuffle=True,  drop_last=True)
    kitti_val_loader   = build_dataloader(kitti_val,   batch_size=FT_BATCH, shuffle=False)
    print(f"[INFO] KITTI train  : {len(kitti_train_subset):,} samples  ({len(kitti_train_loader):,} batches)")
    print(f"[INFO] KITTI val    : {len(kitti_val):,}  samples  ({len(kitti_val_loader):,}  batches)")

    # Sanity check
    _rgb_k, _depth_k = next(iter(kitti_train_loader))
    print(f"[INFO] KITTI batch  : RGB {tuple(_rgb_k.shape)}  Depth {tuple(_depth_k.shape)}")
    print(f"[INFO] Depth range  : [{_depth_k.min():.3f}, {_depth_k.max():.3f}] m")
    assert _rgb_k.shape[1:]   == (3, 192, 636), f"Expected (3,192,636) — got {_rgb_k.shape[1:]}"
    assert _depth_k.shape[1:] == (1, 192, 636), f"Expected (1,192,636) — got {_depth_k.shape[1:]}"
    print("[OK] KITTI shape verified")

print("\n[OK] Dataset ready")

# 5. Network
This section structures the neural network. The neural network model architecture `METERModel` (comprising a MobileViT encoder and a custom decoder) is defined in `src/network.py` and imported.

In [ ]:
# =====================================================================
# 5. Network
# =====================================================================
# Instantiate the core METERModel class and verify shapes via dummy input
# Instantiate the METER model to verify the structure
model = METERModel(arch_type=ARCH_TYPE).to(DEVICE)

# Verify network input/output shape (resolution depends on the selected dataset)
_net_h, _net_w = NYU_SIZE if DATASET_NAME == "nyu" else KITTI_SIZE
dummy_input = torch.randn(1, 3, _net_h, _net_w).to(DEVICE)
with torch.no_grad():
    out = model(dummy_input)
print(f"\n[OK] Network structure successfully verified. Output shape: {out.shape}")

# 6. Train
This section contains the training cycle elements. It includes both the Self-Supervised LeJEPA pre-training and the Supervised METER Fine-Tuning transfer learning pipeline.

**Baseline choice.** The original METER initializes its encoder with ImageNet-supervised weights; no public ImageNet checkpoint exists for this custom MobileViT-lite encoder, so the controlled comparison is **LeJEPA pre-training vs. random initialization** on the *same* architecture, optimizer, schedule and data. This isolates the contribution of the SSL objective itself (Project 7, Objective 3).

In [ ]:
# =====================================================================
# 6A. Self-Supervised Learning Pre-Training (LeJEPA SSL)
# =====================================================================
# Build the LeJEPA training module and fit the network on unlabeled raw data
# Parameters (override config.py if different)
SSL_ARCH           = ARCH_TYPE          # "xxs" | "xs" | "s"
SSL_EPOCHS         = EPOCHS_PRETRAIN    # 50
SSL_BATCH          = SSL_BATCH_SIZE     # 16
SKIP_PRETRAIN      = False              # True → skip, load existing ckpt

ssl_ckpt_dir = MODEL_SAVE_PATH / f"ssl_{DATASET_NAME}"
ssl_ckpt_dir.mkdir(parents=True, exist_ok=True)

# 6A.1: LeJEPA Model 
print("[STEP 6A.1] Building LeJEPA model")
ssl_model = LeJEPAMeter(
    arch_type     = SSL_ARCH,
    proj_dim      = SSL_PROJ_DIM,
    num_slices    = SSL_NUM_SLICES,
    lamb          = SSL_LAMB,
    lr            = SSL_LR,
    weight_decay  = SSL_WEIGHT_DECAY,
    warmup_epochs = 10,
    dataset       = DATASET_NAME,
)

# Verify encoder → projector communication
# MobileViT encoder returns (features, [y0, y1, y2, y3])
# LeJEPAMeter does: features, _ = encoder(x) → emb = features.mean([-2,-1])
_dummy = torch.randn(2, 3, 256, 256)
_emb, _proj = ssl_model(_dummy)
print(f"[INFO] Encoder output (emb)  : {tuple(_emb.shape)} — expected (2, {cfg.ENC_DIMS.get(SSL_ARCH, '?')})")
print(f"[INFO] Projector output (proj): {tuple(_proj.shape)} — expected (2, {SSL_PROJ_DIM})")
assert _proj.shape == (2, SSL_PROJ_DIM), "Projector shape mismatch!"
print("[OK] Encoder → Projector: communication verified")

# 6A.2: Callbacks and Logger 
ssl_callbacks = [
    ModelCheckpoint(
        dirpath   = str(ssl_ckpt_dir),
        filename  = "ssl-{epoch:02d}-{val/loss:.4f}",
        monitor   = "val/loss",
        mode      = "min",
        save_top_k = 1,
        save_last  = True,
    ),
    # EarlyStopping(
    #    monitor  = "val/loss",
    #    patience = 10,
    #    mode     = "min",
    #    strict   = False,
    #),
    LearningRateMonitor(logging_interval="epoch"),
]
ssl_logger = CSVLogger(save_dir=str(OUTPUT_DIR), name="ssl_log")

# 6A.3: Trainer 
ssl_trainer = L.Trainer(
    max_epochs         = SSL_EPOCHS,
    accelerator        = "auto",
    devices            = 1,
    precision="16-mixed",
    gradient_clip_val       = 1.0,        
    gradient_clip_algorithm = "norm",
    callbacks          = ssl_callbacks,
    logger             = ssl_logger,
    log_every_n_steps  = 10,
    enable_progress_bar = True,
    accumulate_grad_batches = 1,         # fix SIGReg covariance instability
)

# 6A.4: Training 
if SKIP_PRETRAIN:
    print("[INFO] SKIP_PRETRAIN=True — loading existing checkpoint")
    _ckpt = str(ssl_ckpt_dir / "last.ckpt")
    if not Path(_ckpt).exists():
        print("[WARN] No SSL checkpoint found — fine-tuning will start from scratch")
else:
    print(f"\n[STEP 6A.2] Starting SSL pre-training — {SSL_EPOCHS} epochs")
    _t = Timer()

    # Resume from checkpoint if it exists
    _last_ssl = ssl_ckpt_dir / "last.ckpt"
    ssl_trainer.fit(
        ssl_model,
        train_dataloaders = ssl_loader,
        val_dataloaders   = ssl_val_loader,
        ckpt_path         = str(_last_ssl) if _last_ssl.exists() else None,
    )
    print(f"[INFO] Pre-training completed in {_t.end()}")
    print(f"[INFO] SSL checkpoint → {ssl_ckpt_dir / 'last.ckpt'}")

# 6A.5: Zero-shot Geometric Probing 
print("\n[STEP 6A.3] Zero-shot Geometric Probing (PCA)")
_probe_loader = ssl_val_loader
geometric_probing_test_set(
    model       = ssl_model,
    device      = DEVICE,
    test_loader = _probe_loader,
    num_samples = 4,
    save_path   = str(IMAGES_DIR / "pca_ssl_encoder.png"),
)
print("[OK] SSL probing completed")

In [ ]:
# =====================================================================
# 6B. Supervised Fine-Tuning (MDE Transfer Learning)
# =====================================================================
# Fine-tune the pre-trained encoder + decoder under a combined MDE loss function
# Parameters
FT_ARCH          = ARCH_TYPE        # must match pre-training
FT_DATASET       = DATASET_NAME     # "nyu" | "kitti"
FT_EPOCHS        = EPOCHS_FINETUNE  # 60 — paper Section IV-A
FREEZE_ENCODER   = False            # True → train only decoder
USE_SSL_CKPT     = True             # True → load encoder from LeJEPA pre-training

ft_ckpt_dir = MODEL_SAVE_PATH / f"finetune_{FT_DATASET}"
ft_ckpt_dir.mkdir(parents=True, exist_ok=True)

# Path to SSL checkpoint for encoder loading
_ssl_ckpt_path = str(MODEL_SAVE_PATH / f"ssl_{DATASET_NAME}" / "last.ckpt") if USE_SSL_CKPT else None
if USE_SSL_CKPT and not Path(_ssl_ckpt_path).exists():
    print("[WARN] SSL checkpoint not found — fine-tuning will start from random weights")
    _ssl_ckpt_path = None

# 6B.1: Select train/val loaders
if FT_DATASET == "nyu":
    ft_train_loader = nyu_train_loader
    ft_val_loader   = nyu_test_loader
    val_monitor     = "val/rmse"
elif FT_DATASET == "kitti":
    ft_train_loader = kitti_train_loader
    ft_val_loader   = kitti_val_loader
    val_monitor     = "val/rmse"
else:
    raise ValueError(f"FT_DATASET must be 'nyu' or 'kitti', not '{FT_DATASET}'")

# 6B.2: METERFinetune Model
# NYU: 1e-3 as in METER paper (Sec. IV-A).
# KITTI: 1e-4 — empirically required for stability (1e-3 diverges on metric-scale depth).
ft_lr = LEARNING_RATE if FT_DATASET == "nyu" else 1e-4

ft_model = METERFinetune(
    arch_type       = FT_ARCH,
    lr              = ft_lr,
    weight_decay    = WEIGHT_DECAY,
    pretrained_ckpt = _ssl_ckpt_path,
    freeze_encoder  = FREEZE_ENCODER,
    dataset_name    = FT_DATASET,
)

# Verify encoder → decoder communication
# MobileViT.forward() → (bottleneck, [y0, y1, y2, y3])
# Decoder.forward(bottleneck, [y0, y1, y2, y3]) → depth map
print("[INFO] Verifying encoder → decoder (skip connections)")
_h, _w = (192, 256) if FT_DATASET == "nyu" else (192, 636)
_dummy_ft = torch.randn(1, 3, _h, _w)
_enc_out, _skips = ft_model.model.encoder(_dummy_ft)
print(f"  Encoder bottleneck : {tuple(_enc_out.shape)}")
for i, sk in enumerate(_skips):
    print(f"  Skip y{i}          : {tuple(sk.shape)}")
_dec_out = ft_model.model.decoder(_enc_out, _skips)
print(f"  Decoder output     : {tuple(_dec_out.shape)} — expected (1, 1, {_h//32 * 8}, ...)")
_full_out = ft_model(_dummy_ft)
print(f"  Full model output  : {tuple(_full_out.shape)} — (after interpolate → {_h}×{_w})")
assert _full_out.shape == (1, 1, _h, _w), f"Output shape mismatch: {_full_out.shape}"
print("[OK] Encoder → Decoder: communication verified")

# 6B.3: Callbacks and Logger 
ft_callbacks = get_callbacks(
    save_dir     = str(ft_ckpt_dir),
    monitor      = val_monitor,
    dataset_name = FT_DATASET,
    arch_name    = FT_ARCH,
)
ft_logger = CSVLogger(save_dir=str(OUTPUT_DIR), name=f"finetune_{FT_DATASET}_log")

# 6B.4: Trainer
ft_trainer = L.Trainer(
    max_epochs              = FT_EPOCHS,
    accelerator             = "auto",
    devices                 = 1,
    precision               = "32",
    callbacks               = ft_callbacks,
    logger                  = ft_logger,
    log_every_n_steps       = 10,
    enable_progress_bar     = True,
    accumulate_grad_batches = 8 if FT_DATASET == "kitti" else 1,
    gradient_clip_val       = 5.0 if FT_DATASET == "kitti" else 1.0,
    gradient_clip_algorithm = "norm",
)

# 6B.5: Training
print(f"\n[STEP 6B.1] Starting METER-{FT_ARCH.upper()} fine-tuning on {FT_DATASET.upper()}")
print(f"  Epochs     : {FT_EPOCHS}  (paper Section IV-A)")
print(f"  Initial LR : {ft_lr}  → /10 at epochs 20, 40")
print(f"  Loss       : L_depth + {LAMBDA_1}*L_grad + {LAMBDA_2}*L_norm + {LAMBDA_3}*L_SSIM")

_t_ft = Timer()

# Resume from checkpoint if it exists
_last_ft = ft_ckpt_dir / "last.ckpt"
ft_trainer.fit(
    ft_model,
    train_dataloaders = ft_train_loader,
    val_dataloaders   = ft_val_loader,
    ckpt_path         = str(_last_ft) if _last_ft.exists() else None,
)
print(f"[INFO] Fine-tuning completed in {_t_ft.end()}")

# 6B.6: Final test
print(f"\n[STEP 6B.2] Final test on {FT_DATASET.upper()}")
ft_trainer.test(ft_model, dataloaders=ft_val_loader)

# 6B.7: Post fine-tuning PCA probing
print("\n[STEP 6B.3] Post fine-tuning Geometric Probing")
geometric_probing_test_set(
    model       = ft_model,
    device      = DEVICE,
    test_loader = ft_val_loader,
    num_samples = 4,
    save_path   = str(IMAGES_DIR / f"pca_finetune_{FT_DATASET}.png"),
)
print("[OK] Fine-tuning probing completed")

# Save final metrics to JSON
from src.utils import save_dict_to_json
_test_results = ft_trainer.logged_metrics
_out = {
    "arch"           : FT_ARCH,
    "dataset"        : FT_DATASET,
    "epochs"         : FT_EPOCHS,
    "ssl_pretrained" : USE_SSL_CKPT,
    "freeze_encoder" : FREEZE_ENCODER,
}
_out.update({k: float(v) for k, v in _test_results.items() if isinstance(v, torch.Tensor)})
save_dict_to_json(_out, JSON_DIR / f"results_{FT_DATASET}_{FT_ARCH}.json")

print("\n" + "=" * 55)
print("  Pipeline completed.")
print(f"  Checkpoint → {ft_ckpt_dir}")
print(f"  Results    → {JSON_DIR}")
print(f"  Images     → {IMAGES_DIR}")
print("=" * 55)

import gc
import torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
# =====================================================================
# 6C. From-Scratch Training Baseline (Encoder randomly initialized)
# =====================================================================
# Control experiment: train METER from scratch (no SSL) for comparative analysis
# Parameters
FT_ARCH          = ARCH_TYPE        # must match pre-training
FT_DATASET       = DATASET_NAME     # "nyu" | "kitti"
FT_EPOCHS        = EPOCHS_FINETUNE  # 60 — paper Section IV-A
FREEZE_ENCODER   = False            # True → train only decoder
USE_SSL_CKPT     = False            # False → do not load encoder from LeJEPA pre-training



L.seed_everything(42, workers=True)

fs_ckpt_dir = MODEL_SAVE_PATH / f"finetune_from_scratch_{FT_DATASET}"
fs_ckpt_dir.mkdir(parents=True, exist_ok=True)

# 6C.1
if FT_DATASET == "nyu":
    fs_train_loader, fs_val_loader = nyu_train_loader, nyu_test_loader
elif FT_DATASET == "kitti":
    fs_train_loader, fs_val_loader = kitti_train_loader, kitti_val_loader
else:
    raise ValueError(f"FT_DATASET must be 'nyu' or 'kitti', not '{FT_DATASET}'")
val_monitor = "val/rmse"

# 6C.2
print(f"\n[STEP 6C.1] Building from-scratch baseline (arch={FT_ARCH}, dataset={FT_DATASET})")
print("[INFO] pretrained_ckpt=None  →  encoder RANDOMLY initialized")
fs_lr = LEARNING_RATE if FT_DATASET == "nyu" else 1e-4   # same LR policy as the SSL-pretrained run (fair comparison)
fs_model = METERFinetune(
    arch_type       = FT_ARCH,
    lr              = fs_lr,
    weight_decay    = WEIGHT_DECAY,
    pretrained_ckpt = None, 
    freeze_encoder  = False,
    dataset_name    = FT_DATASET,
)

# 6C.3: Callbacks 
fs_callbacks = get_callbacks(
    save_dir     = str(fs_ckpt_dir),
    monitor      = val_monitor,
    dataset_name = FT_DATASET,
    arch_name    = FT_ARCH,
)
fs_logger = CSVLogger(save_dir=str(OUTPUT_DIR), name=f"finetune_from_scratch_{FT_DATASET}_log")

# 6C.4: Trainer 
fs_trainer = L.Trainer(
    max_epochs              = FT_EPOCHS,
    accelerator             = "auto",
    devices                 = 1,
    precision               = "32",
    callbacks               = fs_callbacks,
    logger                  = fs_logger,
    log_every_n_steps       = 10,
    enable_progress_bar     = True,
    accumulate_grad_batches = 8 if FT_DATASET == "kitti" else 1,
    gradient_clip_val       = 5.0 if FT_DATASET == "kitti" else 1.0,
    gradient_clip_algorithm = "norm",
)

# 6C.5: Training
print(f"\n[STEP 6C.2] Starting from-scratch baseline on {FT_DATASET.upper()}")
print(f"  Epochs     : {FT_EPOCHS}")
print(f"  Initial LR : {fs_lr}")
print(f"  Loss       : L_depth + {LAMBDA_1}*L_grad + {LAMBDA_2}*L_norm + {LAMBDA_3}*L_SSIM")
print(f"  Output dir : {fs_ckpt_dir}")

_t_fs = Timer()
_last_fs = fs_ckpt_dir / "last.ckpt"
fs_trainer.fit(
    fs_model,
    train_dataloaders = fs_train_loader,
    val_dataloaders   = fs_val_loader,
    ckpt_path         = str(_last_fs) if _last_fs.exists() else None,
)
print(f"[INFO] From-scratch training completed in {_t_fs.end()}")

# 6C.6: Final test
print(f"\n[STEP 6C.3] Final test (baseline) on {FT_DATASET.upper()}")
fs_trainer.test(fs_model, dataloaders=fs_val_loader)

# 6C.7: PCA probing 
print("\n[STEP 6C.4] Geometric Probing — from-scratch encoder")
geometric_probing_test_set(
    model       = fs_model,
    device      = DEVICE,
    test_loader = fs_val_loader,
    num_samples = 4,
    save_path   = str(IMAGES_DIR / f"pca_finetune_from_scratch_{FT_DATASET}.png"),
)
print("[OK] Baseline probing completed")

# 6C.8
from src.utils import save_dict_to_json
_test_results_fs = fs_trainer.logged_metrics
_out_fs = {
    "arch"           : FT_ARCH,
    "dataset"        : FT_DATASET,
    "epochs"         : FT_EPOCHS,
    "ssl_pretrained" : False,
    "freeze_encoder" : False,
    "variant"        : "from_scratch_baseline",
}
_out_fs.update({k: float(v) for k, v in _test_results_fs.items() if isinstance(v, torch.Tensor)})
save_dict_to_json(_out_fs, JSON_DIR / f"results_from_scratch_main_{FT_DATASET}_{FT_ARCH}.json")

print("\n" + "=" * 55)
print("  Baseline (from-scratch) run completed.")
print(f"  Checkpoint → {fs_ckpt_dir}")
print(f"  Results    → {JSON_DIR / f'results_from_scratch_{FT_DATASET}_{FT_ARCH}.json'}")
print(f"  Images     → {IMAGES_DIR / f'pca_finetune_from_scratch_{FT_DATASET}.png'}")
print("=" * 55)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# 7. Evaluation
This section performs the tests and evaluations needed for the trained network, including test-set metric evaluation and geometric probing (PCA visualization).

Quantitative comparison (RMSE, AbsRel, δ₁) between the LeJEPA-pretrained model and the from-scratch baseline. Convergence-speed curves (Objective 3) and the efficiency–accuracy trade-off, latency/FPS benchmark and training-cost analysis (Objective 4) are produced in `analysis.ipynb`.

In [ ]:
# =====================================================================
# 7. Evaluation
# =====================================================================
# Final quantitative metrics on the target MDE test/val loader.
# NOTE: `pretrained_ckpt=None` prevents __init__ from re-loading SSL encoder
# weights (wasteful + non-portable hardcoded path saved in the hparams);
# Lightning then restores the fine-tuned weights from the checkpoint.

_eval_loader = nyu_test_loader if DATASET_NAME == "nyu" else kitti_val_loader
_results_cmp = {}

# 7A: METER fine-tuned from LeJEPA SSL pre-training
print("=== METER (Pre-trained with LeJEPA SSL) Evaluation ===")
try:
    _ssl_ft_ckpt = find_best_ckpt(MODEL_SAVE_PATH / f"finetune_{DATASET_NAME}")
    if _ssl_ft_ckpt is None:
        raise FileNotFoundError(f"No checkpoint in {MODEL_SAVE_PATH / f'finetune_{DATASET_NAME}'}")
    print(f"[INFO] Checkpoint: {_ssl_ft_ckpt.name}")
    best_ssl_ft_model = METERFinetune.load_from_checkpoint(
        str(_ssl_ft_ckpt),
        pretrained_ckpt = None,
        freeze_encoder  = False,
        dataset_name    = DATASET_NAME,
    ).to(DEVICE)

    test_metrics = evaluate_pixel_weighted(
        model        = best_ssl_ft_model,
        dataloader   = _eval_loader,
        device       = DEVICE,
        dataset_name = DATASET_NAME,
    )
    _results_cmp["LeJEPA"] = test_metrics
    print(f"Pixel-Weighted Metrics: {test_metrics}")
except Exception as e:
    print(f"Could not evaluate best SSL-FT model: {e}")

# 7B: From-scratch baseline
print("\n=== METER (From-scratch Baseline) Evaluation ===")
try:
    _fs_ckpt = find_best_ckpt(MODEL_SAVE_PATH / f"finetune_from_scratch_{DATASET_NAME}")
    if _fs_ckpt is None:
        raise FileNotFoundError(f"No checkpoint in {MODEL_SAVE_PATH / f'finetune_from_scratch_{DATASET_NAME}'}")
    print(f"[INFO] Checkpoint: {_fs_ckpt.name}")
    best_baseline_model = METERFinetune.load_from_checkpoint(
        str(_fs_ckpt),
        pretrained_ckpt = None,
        freeze_encoder  = False,
        dataset_name    = DATASET_NAME,
    ).to(DEVICE)

    test_metrics_base = evaluate_pixel_weighted(
        model        = best_baseline_model,
        dataloader   = _eval_loader,
        device       = DEVICE,
        dataset_name = DATASET_NAME,
    )
    _results_cmp["From-scratch"] = test_metrics_base
    print(f"Pixel-Weighted Baseline Metrics: {test_metrics_base}")
except Exception as e:
    print(f"Could not evaluate baseline model: {e}")

# 7C: Side-by-side comparison (Project 7, Objective 3)
if len(_results_cmp) == 2:
    print("\n" + "=" * 55)
    print(f"  {DATASET_NAME.upper()} — LeJEPA vs From-scratch  (METER-{ARCH_TYPE.upper()})")
    print("=" * 55)
    _keys = sorted(set(_results_cmp["LeJEPA"]) & set(_results_cmp["From-scratch"]))
    print(f"  {'Metric':<12} {'LeJEPA':>10} {'Scratch':>10} {'Δ %':>8}")
    for k in _keys:
        a, b = float(_results_cmp["LeJEPA"][k]), float(_results_cmp["From-scratch"][k])
        delta = (a - b) / b * 100 if b != 0 else float("nan")
        print(f"  {k:<12} {a:>10.4f} {b:>10.4f} {delta:>+7.1f}%")
    print("=" * 55)

    from src.utils import save_dict_to_json
    save_dict_to_json(
        {k: {m: float(v) for m, v in d.items()} for k, d in _results_cmp.items()},
        JSON_DIR / f"comparison_{DATASET_NAME}_{ARCH_TYPE}.json",
    )
    print(f"[INFO] Comparison saved → {JSON_DIR / f'comparison_{DATASET_NAME}_{ARCH_TYPE}.json'}")
